# 06. 데이터셋 합성 — **Colab GPU**

대본 → TTS → 발화 타임스탬프 → 질의/정답 파일까지.

**`MODE`** (셀 2): `"pilot"` = 설계 검증용 20클립, `"main"` = 본 데이터셋 1,300클립 (train 1000 / dev 100 / test 200).
클립·목소리·트랙 노브는 `kbolisten/build.py` — 07·08과 같은 MODE면 같은 클립이 나온다.

- 파일럿 v2: Hard = 판정 생략(K1) + 드문 이벤트 우선 (v1은 태그만으로 Hard → 베이스라인에서 Easy보다 쉬웠음)
- v3: 해설위원 추가. Hard에만 방해 코멘트. 질의 시각은 다음 발화 0.1초 전
- v4: 07에서 LLM 의역·검증한 대본을 합성. 캐스터 3명(05c) × 해설 2명, 타석 종료 뒤 해설 코멘트 증가
- v5: 의역은 Hard만 (v4는 두 트랙 다 의역해서 트랙 차이가 사라졌음)

**이어서 합성**: 클립마다 음성(`audio/`)과 발화 시각(`clips/`)을 바로 저장한다.
런타임이 끊기면 같은 셀들을 다시 실행하면 된다 — 이미 만든 클립은 건너뛴다.
본 데이터(약 35시간 분량)는 A100·G4 기준 약 8시간. L4는 배치를 줄여서 돌아가지만 2배쯤 느리다.

**실행 전 준비**
- 새 런타임 권장 (07의 vLLM이 PyTorch 버전을 바꿔서 qwen-tts와 꼬일 수 있음)
- Drive `내 드라이브/kbo-listen/`에 `labels_all.jsonl`(03), `voices/`(05b·05c), `paraphrase_{pilot|main}/scripts.jsonl`(07)
- `kbolisten/`은 Drive for desktop으로 동기화한 폴더(`Othercomputers/*/kbolisten`)에서 자동으로 불러온다

**결과물 (Drive `kbo-listen/dataset_{pilot_v5|main}/`)**
| 파일 | 내용 | 공개 대상 |
|---|---|---|
| `audio/{clip_id}.wav` | 합성 중계 음성 | 참가자 |
| `queries.json` | 클립별 질의 시점 t | 참가자 |
| `gt.json` | 각 t의 정답 B-S-O | 평가 코드 (test는 비공개) |
| `scripts.jsonl` | 발화 텍스트·화자 + 시작/끝 시각 (의역한 발화는 원문 `orig` 포함) | train만 공개 (test 대본을 주면 ASR 없이 풀림) |
| `meta.jsonl` | split, 트랙, 태그, 목소리, 노브 값 | 분석용 |
| `clips/{clip_id}.json` | 클립별 발화 시각 (이어서 합성용) | 내부용 |

In [1]:
# 런타임이 새로 시작됐을 때만
!pip install -q -U qwen-tts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 8.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 167.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 160.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the pac

In [2]:
MODE = "main"  # "pilot"(설계 검증 20클립) 또는 "main"(본 데이터셋 1,300클립)

import glob
import json
import os
import random
import sys
import time
import zlib
from collections import Counter
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
from google.colab import drive
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

# kbolisten은 Drive for desktop으로 동기화한 로컬 폴더에서 불러온다 (매번 업로드할 필요 없음)
drive.mount("/content/drive")
_found = glob.glob("/content/drive/Othercomputers/*/kbolisten")
assert _found, "Drive '컴퓨터' 영역에서 kbolisten을 못 찾음 → Drive for desktop 동기화 확인"
sys.path.insert(0, os.path.dirname(_found[0]))

from kbolisten import build as B
from kbolisten.script import queryable, tts_text

# 입력과 결과는 Drive에 둔다 (런타임이 바뀌어도 남도록)
DRIVE = Path("/content/drive/MyDrive/kbo-listen")
OUT = DRIVE / B.DIRS[MODE]["dataset"]
# 07에서 의역·검증한 대본. None이면 여기서 템플릿 대본을 새로 만든다 (의역 없이)
SCRIPTS = DRIVE / B.DIRS[MODE]["paraphrase"] / "scripts.jsonl"
(OUT / "audio").mkdir(parents=True, exist_ok=True)
(OUT / "clips").mkdir(exist_ok=True)  # 클립별 발화 시각 (이어서 합성할 때 다시 읽음)

# 목소리 목록과 참조 문장 (05·05b·05c에서 확정)
VOICE_DIR = DRIVE / "voices"
VOICES = json.loads((VOICE_DIR / "voices.json").read_text(encoding="utf-8"))
print(MODE, "→", OUT.name, "/ 대본:", SCRIPTS.parent.name if SCRIPTS else "템플릿")

with (DRIVE / "labels_all.jsonl").open(encoding="utf-8") as f:
    clips = [json.loads(line) for line in f]
print(len(clips), "클립")


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
Mounted at /content/drive
main → dataset_main / 대본: paraphrase_main
46787 클립


In [3]:
# 1) 클립과 대본 준비 (샘플링·트랙 노브·목소리 배정은 kbolisten/build.py, 07·08과 같은 결과)
if SCRIPTS:
    # 의역 대본 사용: 대본·목소리·노브는 07 결과 그대로, 이벤트(정답)는 labels에서
    by_id = {c["clip_id"]: c for c in clips}
    with SCRIPTS.open(encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    picked = [{**by_id[r["clip_id"]], **{k: r[k] for k in ("track", "knobs", "voices", "utts")}} for r in rows]
else:
    picked = B.prepare(clips, MODE, VOICES)

print(len(picked), "클립", Counter((c["split"], c["track"]) for c in picked))
print("캐스터 배정", Counter(c["voices"]["caster"] for c in picked))
print("해설 배정", Counter(c["voices"]["analyst"] for c in picked))
para = Counter(c["track"] for c in picked for u in c["utts"] if "orig" in u)
print("의역 반영 발화", dict(para), "(easy는 0이어야 함)")

# 파일럿처럼 적을 때만 클립별로 출력
if len(picked) <= 40:
    for c in picked:
        n = len(c["events"])
        kinds = [u["kind"] for u in c["utts"]]
        print(f"{c['split']:>5} {c['track']:>4} {c['clip_id']}  이벤트 {n:>2}  생략 {n - kinds.count('call'):>2}  "
              f"질의 {len(queryable(c['utts'], n)):>2}  해설 {kinds.count('backchannel') + kinds.count('comment'):>2}  "
              f"방해 {kinds.count('distract'):>2}  의역 {sum('orig' in u for u in c['utts']):>2}  "
              f"{c['voices']['caster']}+{c['voices']['analyst']}  {c['tags']}")

1300 클립 Counter({('train', 'easy'): 500, ('train', 'hard'): 500, ('test', 'easy'): 100, ('test', 'hard'): 100, ('dev', 'easy'): 50, ('dev', 'hard'): 50})
캐스터 배정 Counter({'caster_m35_anchor': 449, 'caster_m38_steady': 433, 'caster_m40_baritone': 418})
해설 배정 Counter({'analyst_a45_clear': 653, 'analyst_a40_warm': 647})
의역 반영 발화 {'hard': 12254} (easy는 0이어야 함)


In [4]:
# 3) TTS 모델 + 목소리 등록 (목소리마다 자기 참조 문장으로)
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base", device_map="cuda:0", dtype=torch.bfloat16, attn_implementation="sdpa",
)
prompts = {}
for v, info in VOICES.items():
    ref, ref_sr = sf.read(VOICE_DIR / f"ref_{v}.wav", dtype="float32")
    prompts[v] = model.create_voice_clone_prompt(ref_audio=(ref, ref_sr), ref_text=info["ref_text"])
print("목소리 등록:", list(prompts))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

목소리 등록: ['analyst_a40_warm', 'analyst_a45_clear', 'caster_m38_steady', 'caster_m40_baritone', 'caster_m35_anchor']


In [13]:
# 4) 화자별로 배치 합성 → 원래 순서대로 무음을 넣으며 이어 붙이고, 발화마다 시작/끝 시각 기록
# 클립마다 음성(audio/)과 발화 시각(clips/)을 바로 저장 → 런타임이 끊겨도 다시 실행하면 이어서 한다
LEAD, TAIL = 0.5, 1.0   # 앞뒤 무음(초)
GAP = (0.3, 0.9)        # 발화 사이 무음 범위(초), 클립 ID 시드로 고정
# 한 번에 합성하는 문장 수. 클립 하나의 캐스터 대사(40줄 이상)를 통째로 넣으면 L4(24GB)에서 OOM
# 배치를 나눠도 문장별 음성은 같고 속도만 달라진다
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH = 32 if GPU_GB > 40 else 8
print(f"GPU {GPU_GB:.0f}GB → 배치 {BATCH}")


def silence(sec, sr):
    return np.zeros(int(sec * sr), dtype=np.float32)


def synth_clip(c):
    texts = [tts_text(u["text"]) for u in c["utts"]]
    wavs = [None] * len(texts)
    for role in ("caster", "analyst"):
        idx = [i for i, u in enumerate(c["utts"]) if u["speaker"] == role]
        for k in range(0, len(idx), BATCH):
            part = idx[k:k + BATCH]
            out, sr = model.generate_voice_clone(
                text=[texts[i] for i in part], language=["Korean"] * len(part),
                voice_clone_prompt=prompts[c["voices"][role]],
            )
            for i, w in zip(part, out):
                wavs[i] = np.asarray(w, dtype=np.float32)

    rng = random.Random(zlib.crc32(c["clip_id"].encode()))
    pieces = [silence(LEAD, sr)]
    n = len(pieces[0])  # 지금까지 이어 붙인 샘플 수 → 시각은 n / sr로 계산 (누적 오차 없음)
    warnings = []
    for u, text, w in zip(c["utts"], texts, wavs):
        dur = len(w) / sr
        # TTS가 말을 지어내거나(너무 김) 끊어먹는(너무 짧음) 경우 표시
        if dur > 0.35 * len(text) + 1.0 or dur < 0.03 * len(text):
            warnings.append(f"{dur:.1f}초 | {text}")
        u["start"] = round(n / sr, 3)
        pieces.append(w)
        n += len(w)
        u["end"] = round(n / sr, 3)
        gap = silence(rng.uniform(*GAP), sr)
        pieces.append(gap)
        n += len(gap)
    pieces.append(silence(TAIL, sr))
    return np.concatenate(pieces), sr, warnings


done = skipped = 0
total_audio = total_gen = 0.0
t_start = time.perf_counter()
for k, c in enumerate(picked):
    wav_path = OUT / "audio" / f"{c['clip_id']}.wav"
    cache = OUT / "clips" / f"{c['clip_id']}.json"
    if wav_path.exists() and cache.exists():  # 이미 합성한 클립: 발화 시각만 다시 읽는다
        saved = json.loads(cache.read_text(encoding="utf-8"))
        c["utts"], c["duration"] = saved["utts"], saved["duration"]
        skipped += 1
        continue
    t0 = time.perf_counter()
    audio, sr, warnings = synth_clip(c)
    gen = time.perf_counter() - t0
    c["duration"] = round(len(audio) / sr, 3)
    sf.write(wav_path, audio, sr)
    cache.write_text(json.dumps({"utts": c["utts"], "duration": c["duration"]}, ensure_ascii=False), encoding="utf-8")
    done += 1
    total_audio += c["duration"]
    total_gen += gen
    for w in warnings:
        print(f"   ⚠ {c['clip_id']} {w}")
    if done % 20 == 0 or len(picked) <= 40:
        left = len(picked) - k - 1
        eta = (time.perf_counter() - t_start) / done * left / 60
        print(f"{k + 1}/{len(picked)}  {c['clip_id']}: {c['duration']:5.1f}초 음성, {gen:5.1f}초 걸림  (남은 예상 {eta:.0f}분)")

print(f"\n새로 합성 {done}개, 건너뜀 {skipped}개")
if total_audio:
    print(f"이번 실행: 음성 {total_audio / 60:.1f}분 / 합성 {total_gen / 60:.1f}분 → RTF {total_gen / total_audio:.2f}")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


GPU 95GB → 배치 32


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



새로 합성 3개, 건너뜀 1297개
이번 실행: 음성 6.5분 / 합성 0.4분 → RTF 0.06


In [14]:
# 5) 질의 시점과 정답
# 이벤트 i의 정보가 다 전달된 뒤(그 이벤트에 딸린 마지막 발화의 끝), 다음 발화가 시작되기 0.1초 전에 묻는다
#  → ASR 단어 시각이 실제보다 늦게 찍혀도 최대한 여유를 준다 (발화 사이 간격은 최소 0.3초)
# 판정 생략 때문에 아직 알 수 없는 이벤트는 묻지 않는다 (queryable)
QUERY_LEAD = 0.1


def make_gt(c):
    last_end = {}
    for u in c["utts"]:
        if u["event_idx"] is not None:
            last_end[u["event_idx"]] = u["end"]  # 판정 → 재언급 → 이닝 종료 순이라 마지막 값이 최대
    starts = [u["start"] for u in c["utts"]]

    gt = []
    for i in queryable(c["utts"], len(c["events"])):
        e, end = c["events"][i], last_end[i]
        nxt = next((s for s in starts if s > end), c["duration"])
        t = max(end + QUERY_LEAD, nxt - QUERY_LEAD)
        gt.append({"t": round(t, 3), "b": e["b"], "s": e["s"], "o": e["o"]})
    return gt


gt = {c["clip_id"]: make_gt(c) for c in picked}
queries = {cid: [q["t"] for q in qs] for cid, qs in gt.items()}

# 질의 시각이 클립 안에서 오름차순인지 확인
for c in picked:
    ts = queries[c["clip_id"]]
    assert ts == sorted(ts) and ts[-1] <= c["duration"], c["clip_id"]
for track in ("easy", "hard"):
    print(track, "질의", sum(len(queries[c["clip_id"]]) for c in picked if c["track"] == track), "개")

easy 질의 10413 개
hard 질의 10433 개


In [15]:
# 6) 저장
def dump_json(obj, name):
    (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding="utf-8")


dump_json(queries, "queries.json")
dump_json(gt, "gt.json")

with (OUT / "scripts.jsonl").open("w", encoding="utf-8") as f:
    for c in picked:
        f.write(json.dumps({"clip_id": c["clip_id"], "utts": c["utts"]}, ensure_ascii=False) + "\n")

with (OUT / "meta.jsonl").open("w", encoding="utf-8") as f:
    for c in picked:
        meta = {k: c[k] for k in ("clip_id", "split", "track", "tags", "voices", "n_pitches", "duration",
                                  "season", "inning", "half", "away", "home", "knobs")}
        f.write(json.dumps(meta, ensure_ascii=False) + "\n")

print(sorted(p.name for p in OUT.iterdir()))

['audio', 'clips', 'gt.json', 'meta.jsonl', 'queries.json', 'scripts.jsonl']


In [11]:
# 7) 눈과 귀로 확인: Hard 클립 하나의 대본·시각·질의. 판정이 생략된 이벤트는 (생략)으로 표시
c = next(c for c in picked if c["track"] == "hard" and c["split"] == "test")
print(c["clip_id"], c["tags"], c["voices"])

called = {u["event_idx"] for u in c["utts"] if u["kind"] == "call"}
q_iter = iter(gt[c["clip_id"]])
q = next(q_iter, None)
prev_event = -1
for u in c["utts"]:
    i = u["event_idx"]
    if i is not None:
        # 이 발화 전에 말없이 지나간 투구 (재언급만 있고 판정은 생략된 이벤트 자신도 포함)
        for j in range(prev_event + 1, i + 1):
            if j not in called:
                e = c["events"][j]
                print(f"{'':19}  {'(생략)':>11}  [{e['result']}] → {e['b']}-{e['s']}")
        prev_event = max(prev_event, i)
    who = "해설" if u["speaker"] == "analyst" else "캐스터"
    mark = "  [질의X]" if u.get("query") is False else ""
    print(f"{u['start']:6.2f}~{u['end']:6.2f} {who:<3} {u['kind']:>11}  {u['text']}{mark}")
    while q and u["end"] < q["t"] < u["end"] + 1.0:
        print(f"{'':19}  ▶ 질의 t={q['t']:.2f}  정답 {q['b']}-{q['s']}  {q['o']}아웃")
        q = next(q_iter, None)

display(Audio(str(OUT / "audio" / f"{c['clip_id']}.wav")))

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# (기록) main 첫 실행에서 TTS 이상 경고(⚠)가 뜬 클립. 19.9초짜리는 짧은 문장을 반복·지어낸 것
# 지우고 셀 5(합성) → 6 → 7을 다시 실행하면 이 클립들만 새로 합성된다 (TTS는 실행마다 결과가 달라짐)
BAD = ["20250819WOHT02025_03T", "20240329SKSS02024_04B", "20230611LGHH02023_07T"]
for cid in BAD:
    (OUT / "audio" / f"{cid}.wav").unlink(missing_ok=True)
    (OUT / "clips" / f"{cid}.json").unlink(missing_ok=True)
print("삭제 완료")